In [1]:
# Utility function to ensure DataFrames with geometry are converted to GeoDataFrames
import geopandas as gpd
import pandas as pd
from shapely import wkt

def ensure_geodataframe(df, geometry_col='geometry'):
    """
    Ensures that a DataFrame with geometry column is converted to a GeoDataFrame.
    If conversion fails, tries to robustly decode geometry values before failing.
    
    Args:
        df: DataFrame or GeoDataFrame
        geometry_col: Name of the geometry column
    
    Returns:
        GeoDataFrame with proper CRS set
    """
    import shapely
    import binascii

    def try_decode_geometry(val):
        """
        Try to decode a geometry value that may be:
        - Already a shapely geometry
        - A WKT string
        - A WKB hex string (bytes or str)
        - A WKB bytes object
        If it cannot be decoded, returns None.
        """
        if isinstance(val, shapely.geometry.base.BaseGeometry):
            return val
        if val is None or (isinstance(val, float) and pd.isna(val)):
            return None
        # Try WKT
        if isinstance(val, str):
            try:
                # Try WKT first
                return shapely.wkt.loads(val)
            except Exception:
                pass
            try:
                # Try WKB hex string
                return shapely.wkb.loads(binascii.unhexlify(val))
            except Exception:
                pass
        # Try WKB bytes
        if isinstance(val, (bytes, bytearray)):
            try:
                return shapely.wkb.loads(val)
            except Exception:
                pass
        return None

    # If already a GeoDataFrame, ensure CRS is set
    if isinstance(df, gpd.GeoDataFrame):
        if df.crs is None:
            df = df.set_crs(epsg=4326)  # WGS84
        return df

    # If regular DataFrame with geometry column, convert to GeoDataFrame
    if geometry_col in df.columns:
        # Convert geometry column from WKT strings to geometry objects if needed
        if df[geometry_col].dtype == 'object':
            try:
                df[geometry_col] = df[geometry_col].apply(wkt.loads)
            except Exception:
                # If wkt.loads fails, try to robustly decode geometry values
                try:
                    df[geometry_col] = df[geometry_col].apply(try_decode_geometry)
                    # Remove rows where geometry could not be decoded
                    n_invalid = df[geometry_col].isna().sum()
                    if n_invalid > 0:
                        print(f"⚠️ {n_invalid} rows had invalid geometry and will be dropped.")
                        df = df[df[geometry_col].notna()]
                except Exception as e:
                    print(f"❌ Could not decode geometry: {e}")
                    raise

        # Convert to GeoDataFrame
        try:
            df = gpd.GeoDataFrame(df, geometry=geometry_col)
        except Exception as e:
            # Try to robustly decode geometry and try again
            try:
                df[geometry_col] = df[geometry_col].apply(try_decode_geometry)
                n_invalid = df[geometry_col].isna().sum()
                if n_invalid > 0:
                    print(f"⚠️ {n_invalid} rows had invalid geometry and will be dropped.")
                    df = df[df[geometry_col].notna()]
                df = gpd.GeoDataFrame(df, geometry=geometry_col)
            except Exception as e2:
                print(f"❌ Could not convert to GeoDataFrame after robust decode: {e2}")
                raise

        # Set CRS if not already set
        if df.crs is None:
            df = df.set_crs(epsg=4326)  # WGS84

        return df

    # Return as-is if no geometry column found
    return df

print("✅ Utility function loaded: ensure_geodataframe()")

import sys
import pandas as pd
sys.path.append('..')  # Add parent directory to path
from cloud_utils import get_feature_data, get_feature_data_with_geometry
scrape_data = 0

import os
import glob
from datetime import datetime

# Directory to save/load data
data_dir = "data/morgantown/"
os.makedirs(data_dir, exist_ok=True)

scrape_data = 1


✅ Utility function loaded: ensure_geodataframe()


In [2]:
if scrape_data == 1:
    print("🔄 Scraping fresh Morgantown property data...")

    # Define the URL for the feature service (Morgantown/Monongalia parcels)
    base_url = "https://services3.arcgis.com/MGBEQZtkTlJin8UB/arcgis/rest/services/Monongalia_AGOL/FeatureServer/6/query"

    # Parameters for the query (following the pattern in the syracuse code)
    params = {
        'where': '1=1',
        'outFields': '*',
        'returnGeometry': 'true',
        'f': 'geojson',
        'resultRecordCount': 1000,
        'resultOffset': 0
    }

    # Determine the total count from the ArcGIS REST API using REST endpoint
    from urllib.request import urlopen
    import json

    count_url = (
        f"{base_url}"
        "?where=1=1&returnCountOnly=true&f=json"
    )
    with urlopen(count_url) as response:
        count_json = json.load(response)
        total_records = count_json['count']
    print(f"\nTotal records in 6/query: {total_records:,}\n")

    # Download records in pages, as in @syracuse.ipynb
    all_features = []
    print("📥 Downloading Morgantown/Monongalia parcel data...")
    while True:
        # Make the request for this page (use requests.get compatible with geojson)
        import requests
        response = requests.get(base_url, params=params)
        geojson_data = response.json()
        features = geojson_data.get('features', [])
        all_features.extend(features)

        print(f"   Downloaded {len(all_features):,} parcels so far...")

        # Check if there are more features to fetch
        if len(features) < params['resultRecordCount']:
            break

        params['resultOffset'] += params['resultRecordCount']

    # Convert to GeoDataFrame
    import geopandas as gpd
    gdf = gpd.GeoDataFrame.from_features(all_features)
    print(f"\n✅ Downloaded {len(gdf):,} Morgantown/Monongalia parcels")
    print(f"   Columns: {len(gdf.columns)}\n")

    # Ensure gdf is a proper GeoDataFrame before saving
    gdf = ensure_geodataframe(gdf)

    # Save the processed data with today's date in the filename
    today_str = datetime.now().strftime("%Y%m%d")
    save_path_parquet = os.path.join(data_dir, f"morgantown_parcels_processed_{today_str}.parquet")
    gdf.to_parquet(save_path_parquet)
    print(f"💾 Saved processed data to {save_path_parquet}")

    # Also save as a geopackage
    save_path_gpkg = os.path.join(data_dir, f"morgantown_parcels_processed_{today_str}.gpkg")
    gdf.to_file(save_path_gpkg, driver="GPKG", layer="parcels", index=False)
    print(f"💾 Also saved processed data as geopackage to {save_path_gpkg}")

else:
    print("📂 Loading existing Morgantown property data...")
    # Find all processed parquet files in the data_dir
    parquet_files = glob.glob(os.path.join(data_dir, "morgantown_parcels_processed_*.parquet"))
    if not parquet_files:
        print("❌ No processed Morgantown data files found in data/morgantown/. Please set scrape_data = 1 to download fresh data.")
        raise FileNotFoundError("No processed Morgantown data files found.")
    # Extract dates and find the most recent file
    def extract_date(f):
        try:
            return datetime.strptime(os.path.basename(f).split("_")[-1].replace(".parquet", ""), "%Y%m%d")
        except Exception:
            return datetime.min
    parquet_files_sorted = sorted(parquet_files, key=extract_date, reverse=True)
    most_recent_file = parquet_files_sorted[0]
    import geopandas as gpd
    gdf = gpd.read_parquet(most_recent_file)
    print(f"✅ Loaded processed Morgantown data from {most_recent_file}")

    # Also (re)save as geopackage for convenience
    today_str = datetime.now().strftime("%Y%m%d")
    save_path_gpkg = os.path.join(data_dir, f"morgantown_parcels_processed_{today_str}.gpkg")
    gdf = ensure_geodataframe(gdf)
    gdf.to_file(save_path_gpkg, driver="GPKG", layer="parcels", index=False)
    print(f"💾 Saved (re)loaded data as geopackage to {save_path_gpkg}")

# Ensure gdf is a proper GeoDataFrame
gdf = ensure_geodataframe(gdf)
print(f"\n📊 Dataset Overview:")
print(f"Total parcels: {len(gdf):,}")
print(f"Columns: {len(gdf.columns)}")
print(f"Geometry type: {gdf.geometry.geom_type.iloc[0]}")


import pandas as pd
pd.set_option('display.max_columns', None)
display(gdf.head())



🔄 Scraping fresh Morgantown property data...

Total records in 6/query: 55,864

📥 Downloading Morgantown/Monongalia parcel data...
   Downloaded 1,000 parcels so far...
   Downloaded 2,000 parcels so far...
   Downloaded 3,000 parcels so far...
   Downloaded 4,000 parcels so far...
   Downloaded 5,000 parcels so far...
   Downloaded 6,000 parcels so far...
   Downloaded 7,000 parcels so far...
   Downloaded 8,000 parcels so far...
   Downloaded 9,000 parcels so far...
   Downloaded 10,000 parcels so far...
   Downloaded 11,000 parcels so far...
   Downloaded 12,000 parcels so far...
   Downloaded 13,000 parcels so far...
   Downloaded 14,000 parcels so far...
   Downloaded 15,000 parcels so far...
   Downloaded 16,000 parcels so far...
   Downloaded 17,000 parcels so far...
   Downloaded 18,000 parcels so far...
   Downloaded 19,000 parcels so far...
   Downloaded 20,000 parcels so far...
   Downloaded 21,000 parcels so far...
   Downloaded 22,000 parcels so far...
   Downloaded 23,000

,geometry,OBJECTID,dmp,parid,nbhd,own1,own2,careof,owneraddr,cityname,statecode,legal1,legal2,legal3,book,page,aprland,aprbldg,acres,dist,map,parcel,PRC,IAS_Details,DB_DP,Shape__Area,Shape__Length
0,"MULTIPOLYGON (((-80.40788 39.71771, -80.4085 3...",343561,01-01-1,01 1000100000000,1510,SHIPMAN R ALLAN,,,106 RENNER CREEK RD,NEW FREEPORT,PA,107.50 AC SUR,,,1506,314,105500.0,150200.0,107.50,01,01,1,https://wvias.io/prc/Monongalia/2025/01%20%20%...,https://wvias.io/detail/Monongalia/2025/01%20%...,1506-314,4.911763e+06,16840.497573
1,"MULTIPOLYGON (((-80.41277 39.71668, -80.41326 ...",343562,01-01-2,01 1000200000000,1510,"JOHNSTON LAWRENCE D,",MARY K JOHNSTON,,1017 DEREKTON DR,LATROBE,PA,95 AC SUR,DUNKARD CREEK,,1571,310,70000.0,20900.0,95.00,01,01,2,https://wvias.io/prc/Monongalia/2025/01%20%20%...,https://wvias.io/detail/Monongalia/2025/01%20%...,1571-310,4.378009e+06,11094.094029
2,"MULTIPOLYGON (((-80.41421 39.71553, -80.41423 ...",343563,01-01-3,01 1000300000000,1510,"JOHNSTON LAWRENCE D,",MARY K JOHNSTON,,1017 DEREKTON DR,LATROBE,PA,49.5 AC SUR,,,1571,310,31000.0,0.0,49.50,01,01,3,https://wvias.io/prc/Monongalia/2025/01%20%20%...,https://wvias.io/detail/Monongalia/2025/01%20%...,1571-310,1.885658e+06,7716.098597
3,"MULTIPOLYGON (((-80.40694 39.71697, -80.40513 ...",343564,01-01-4,01 1000400000000,1510,PICHARDO PAUL G (II),,,1188 SAINT CLOUD RD,HUNDRED,WV,90 AC SUR & ALL COG MIRACLE,RUN (API #1572-CNX-CBM ONLY),,1356,490,30800.0,12400.0,90.00,01,01,4,https://wvias.io/prc/Monongalia/2025/01%20%20%...,https://wvias.io/detail/Monongalia/2025/01%20%...,1356-490,3.840050e+06,14200.046363
4,"POLYGON ((-80.40046 39.72126, -80.40109 39.720...",343565,01-01-5,01 1000500000000,1510,SHIPMAN R ALLAN,,,106 RENNER CREEK RD,NEW FREEPORT,PA,28.25 AC SUR,ST CLOUD,,1506,314,29300.0,84600.0,28.25,01,01,5,https://wvias.io/prc/Monongalia/2025/01%20%20%...,https://wvias.io/detail/Monongalia/2025/01%20%...,1506-314,1.393862e+06,5602.424998


### WVIAS API key (required for the cell below)

The value pull uses the WV IAS property-record-card API (`api.wvias.io`), which
requires an `apiKey` query parameter. The key is **not committed to the repo** —
put it in `data/.env` (gitignored):

```
WVIAS_API_KEY=<key>
```

This is not an issued credential: wvias.io's own front end sends the same key
with every visitor's requests, so if the one in `data/.env` stops working, open
wvias.io in a browser, check the network tab for an `api.wvias.io/v1/prc/...`
request, and copy the current `apiKey` value from it (or ask WVIAS for proper
API access).


In [3]:
# Systematically pull 'Land Use' from the API endpoint
import os
import re
from pathlib import Path

def load_wvias_api_key() -> str:
    """WVIAS_API_KEY from the environment, falling back to data/.env (gitignored)."""
    key = os.environ.get("WVIAS_API_KEY", "").strip()
    if key:
        return key
    env_path = Path.cwd().parent / ".env"   # data/jurisidictions/ -> data/.env
    if env_path.exists():
        for line in env_path.read_text(encoding="utf-8").splitlines():
            m = re.match(r"\s*WVIAS_API_KEY\s*=\s*(.+?)\s*$", line)
            if m:
                return m.group(1)
    raise RuntimeError("WVIAS_API_KEY not found in env or data/.env — see the markdown cell above")

WVIAS_API_KEY = load_wvias_api_key()

import requests
from urllib.parse import urlparse, unquote
import re
from tqdm import tqdm

def prc_url_to_api_url(prc_url):
    """
    Convert a PRC link like
    https://wvias.io/prc/Monongalia/2025/01%20%20%201000100000000
    to the API equivalent:
    https://api.wvias.io/v1/prc/Monongalia/2025/01%20%20%201000100000000?apiKey=<WVIAS_API_KEY>
    """
    # Get the part after '/prc/'
    try:
        path = urlparse(prc_url).path
        match = re.match(r"^/prc/([^/]+)/([^/]+)/(.+)$", path)
        if not match:
            return None
        county, year, prc_id = match.groups()
        api_url = (
            f"https://api.wvias.io/v1/prc/{county}/{year}/{prc_id}?apiKey={WVIAS_API_KEY}"
        )
        return api_url
    except Exception:
        return None

def fetch_land_use_from_api(api_url):
    """
    Fetch 'Land Use' from the API JSON.
    """
    try:
        resp = requests.get(api_url, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        # Try multiple possible nestings for 'Land Use'
        # Usually: data["data"][0]["parcel"]["luc"]
        luc = None
        if "data" in data and len(data["data"]) > 0:
            parcel = data["data"][0].get("parcel", {})
            luc = parcel.get("luc")
            # fallback: sometimes as parcel["land"][0]["luc"]
            if (not luc) and "land" in data["data"][0]:
                land_list = data["data"][0]["land"]
                if isinstance(land_list, list) and len(land_list):
                    luc = land_list[0].get("luc") or land_list[0].get("ltype")
            if not luc:
                # Try even deeper: "asmt" maybe
                asmt_list = data["data"][0].get("asmt")
                if isinstance(asmt_list, list) and len(asmt_list):
                    luc = asmt_list[0].get("luc")
        return luc
    except Exception:
        return None

# Pick a small set for demo/validation
test_links = gdf['PRC'].iloc[:2]
land_use_attempts = []

for link in test_links:
    api_url = prc_url_to_api_url(link)
    print(f"PRC link: {link}")
    print(f"API url: {api_url}")
    land_use = fetch_land_use_from_api(api_url)
    print(f"  🏷️ Land Use: {land_use}")
    land_use_attempts.append(land_use)

print("\nSummary of API lookups for first two PRC links:", land_use_attempts)

# To fetch for larger batches, e.g. first 100 or all:
# land_use_all = []
# for link in tqdm(gdf['PRC']):  # tqdm for progress bar
#     api_url = prc_url_to_api_url(link)
#     land_use = fetch_land_use_from_api(api_url)
#     land_use_all.append(land_use)


# Parallel processing in batches of 10, saving to data/morgantown/batches as batches of 1k parcels.
# If process is stopped and skip_existing=True, skip parcels already successfully pulled.

import os
import json
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

BATCH_SIZE = 1000
SUBBATCH_SIZE = 10
BATCH_DIR = "data/morgantown/batches"
SKIP_EXISTING = True  # set to True to skip already pulled parcels

os.makedirs(BATCH_DIR, exist_ok=True)
prc_links = gdf['PRC'].tolist()
total = len(prc_links)

def get_land_use_for_row(prc_link):
    api_url = prc_url_to_api_url(prc_link)
    return fetch_land_use_from_api(api_url)

def batch_filename(batch_idx):
    return os.path.join(BATCH_DIR, f"batch_{batch_idx:04d}.json")

landuse_values = [None] * total

for batch_start in range(0, total, BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, total)
    batch_idx = batch_start // BATCH_SIZE
    batch_file = batch_filename(batch_idx)
    batch_slice = slice(batch_start, batch_end)
    batch_prc_links = prc_links[batch_start:batch_end]
    batch_results = [None] * (batch_end - batch_start)
    already_done = set()

    # If skipping existing, check which parcels in this batch were already pulled
    if SKIP_EXISTING and os.path.exists(batch_file):
        try:
            with open(batch_file, 'r') as f:
                existing = json.load(f)
            # existing is a dict: idx (str) -> land_use
            for i_str, val in existing.items():
                i = int(i_str)
                batch_results[i] = val
                landuse_values[batch_start + i] = val
                if val is not None:
                    already_done.add(i)
        except Exception as e:
            print(f"Warning: failed to load batch file {batch_file}: {e}")

    indices_to_process = [i for i in range(batch_end - batch_start) if i not in already_done]
    if indices_to_process:
        print(f"Processing batch {batch_idx} ({batch_start}-{batch_end}), {len(indices_to_process)} to fetch.")
    else:
        print(f"Skipping batch {batch_idx} ({batch_start}-{batch_end}); already complete.")
        continue

    # Process in subbatches of SUBBATCH_SIZE for parallel API calls
    for sub_start in tqdm(range(0, len(indices_to_process), SUBBATCH_SIZE),
                          desc=f"Batch {batch_idx} progress", leave=False):
        sub_indices = indices_to_process[sub_start:sub_start + SUBBATCH_SIZE]
        # Map indices in batch to absolute indices in gdf
        abs_indices = [batch_start + i for i in sub_indices]
        prc_links_sub = [prc_links[i] for i in abs_indices]

        with ThreadPoolExecutor(max_workers=SUBBATCH_SIZE) as executor:
            future_to_i = {
                executor.submit(get_land_use_for_row, lnk): i_batch
                for i_batch, lnk in zip(sub_indices, prc_links_sub)
            }
            for future in as_completed(future_to_i):
                i_batch = future_to_i[future]
                abs_idx = batch_start + i_batch
                try:
                    land_use = future.result()
                    batch_results[i_batch] = land_use
                    landuse_values[abs_idx] = land_use
                except Exception as e:
                    print(f"Error fetching index {abs_idx}: {e}")
                    batch_results[i_batch] = None
                    landuse_values[abs_idx] = None

        # After each subbatch, save progress
        # Save as dict: index in batch -> land_use
        save_dict = {str(i): batch_results[i] for i in range(len(batch_results)) if batch_results[i] is not None}
        try:
            with open(batch_file, 'w') as f:
                json.dump(save_dict, f)
        except Exception as e:
            print(f"Could not write batch file {batch_file}: {e}")

# Save results to DataFrame
gdf['LandUse'] = landuse_values

completed = sum(1 for v in landuse_values if v is not None)
failed = total - completed

print(f"✅ Fetched LandUse for {completed} rows; {failed} failed or missing.")

# Find all failures and try them one more time

# Find indices where fetching LandUse failed (was None)
failed_indices = [i for i, v in enumerate(landuse_values) if v is None]

print(f"⚠️ Retrying {len(failed_indices)} failed fetches...")

for abs_idx in tqdm(failed_indices, desc="Retrying failures"):
    prc_link = prc_links[abs_idx]
    try:
        land_use = get_land_use_for_row(prc_link)
        landuse_values[abs_idx] = land_use
        gdf.at[abs_idx, 'LandUse'] = land_use
        if land_use is not None:
            print(f"Success at index {abs_idx}")
    except Exception as e:
        print(f"Still failed at index {abs_idx}: {e}")
        continue

# Count remaining failures after retry
still_failed = sum(1 for v in landuse_values if v is None)
print(f"✅ Retry complete. Still missing {still_failed} LandUse values.")


gdf.to_parquet("data/morgantown/parcels_with_landuse.parquet")

import pandas as pd

# Open the written parquet file to verify it saved correctly
gdf_parquet = pd.read_parquet("data/morgantown/parcels_with_landuse.parquet")

# Show the first few rows to confirm it loaded (optional)
display(gdf_parquet.head())

# Find rows where 'dmp' == '07-06B-1' and display them
matching_rows = gdf_parquet[gdf_parquet['dmp'] == '07-06B-1']
display(matching_rows.head())



from shapely.geometry import Polygon
from shapely.ops import unary_union
from shapely.errors import GEOSException

import geopandas as gpd

# Convert DataFrame to GeoDataFrame if needed for spatial index access
def ensure_geodataframe(df):
    if isinstance(df, gpd.GeoDataFrame):
        return df
    elif "geometry" in df.columns:
        # handle geometry dtype if it's not already shapely objects
        if not hasattr(df["geometry"].iloc[0], "geom_type"):
            # Try to convert WKB or WKT
            try:
                from shapely import wkb, wkt
            except ImportError:
                import shapely.wkb as wkb
                import shapely.wkt as wkt
            import numpy as np
            def to_shape(val):
                if isinstance(val, bytes):
                    return wkb.loads(val)
                elif isinstance(val, str):
                    return wkt.loads(val)
                else:
                    return val
            df = df.copy()
            df["geometry"] = df["geometry"].apply(to_shape)
        return gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")
    else:
        raise ValueError("No geometry column present.")

# Compute percent of rows that intersect more than 0.01% of their geometries,
# and print property_use_description counts for intersected parcels
def intersection_percent_and_usage_counts(gdf, threshold=0.0001):
    gdf = ensure_geodataframe(gdf)
    n = len(gdf)
    count_intersected = 0
    intersected_indices = []
    # For speed, use bounding box spatial index first
    sindex = gdf.sindex

    for idx, row in gdf.iterrows():
        geom = row.geometry
        if geom is None or geom.is_empty:
            continue
        possible_matches_index = list(sindex.intersection(geom.bounds))
        # Remove self from possible matches
        possible_matches_index = [i for i in possible_matches_index if i != idx]
        intersects = False
        for other_idx in possible_matches_index:
            other_geom = gdf.iloc[other_idx].geometry
            if other_geom is None or other_geom.is_empty:
                continue
            if not geom.intersects(other_geom):
                continue
            try:
                # Wrap intersection in try/except to catch TopologyException
                inter = geom.intersection(other_geom)
            except (GEOSException, Exception) as e:
                continue
            if inter.is_empty:
                continue
            try:
                frac = inter.area / geom.area
            except Exception:
                continue
            if frac > threshold:
                intersects = True
                break
        if intersects:
            count_intersected += 1
            intersected_indices.append(idx)
    percent = 100 * count_intersected / n if n > 0 else 0
    print(f"Percent of records that intersect another geometry by more than {threshold*100:.3f}% of their area: {percent:.2f}%")
    
    # Print counts of property_use_description for intersected parcels
    if "property_use_description" in gdf.columns:
        desc_counts = gdf.loc[intersected_indices, "property_use_description"].value_counts(dropna=False)
        print("\nproperty_use_description counts for intersected parcels:")
        print(desc_counts)
    else:
        print("'property_use_description' column not found in the dataframe.")
    return percent, intersected_indices

gdf_parquet["property_use_description"] = gdf_parquet["LandUse"]
# Ensure GeoDataFrame before running function
percent_intersected, intersected_indices = intersection_percent_and_usage_counts(gdf_parquet, threshold=0.01)

# Check and explicitly set the Coordinate Reference System (CRS) of the gdf
import geopandas as gpd

print("BEFORE: gdf CRS:", getattr(gdf_parquet, "crs", None))
# If gdf_parquet is not a GeoDataFrame, convert it, assuming 'geometry' is present
if not isinstance(gdf_parquet, gpd.GeoDataFrame):
    # The context provides the ensure_geodataframe function for this purpose
    gdf_parquet = ensure_geodataframe(gdf_parquet)
# If CRS is still not set, set it explicitly
if getattr(gdf_parquet, "crs", None) is None:
    gdf_parquet.set_crs("EPSG:4326", inplace=True, allow_override=True)
print("AFTER: gdf CRS:", gdf_parquet.crs)

# Collapse duplicates on dmp by taking first for aprland/aprbldg and other columns,
# and combining geometries (union if touching, otherwise keep separate in multi)
from shapely.ops import unary_union
import numpy as np

if "dmp" in gdf.columns:
    subset_cols = ["dmp"]

    def collapse_geometries(geoms):
        geoms = [g for g in geoms if g is not None]
        if not geoms:
            return None
        # Combine those that touch/intersect, rest as Multi
        out = []
        while geoms:
            ref = geoms.pop(0)
            group = [ref]
            rest = []
            for g in geoms:
                if ref.intersects(g) or ref.touches(g) or ref.equals(g):
                    group.append(g)
                else:
                    rest.append(g)
            unioned = unary_union(group)
            out.append(unioned)
            geoms = rest
        if len(out) == 1:
            return out[0]
        from shapely.geometry import MultiPolygon
        # Make MultiPolygon if possible
        polygons = []
        for g in out:
            if g.geom_type == "Polygon":
                polygons.append(g)
            elif g.geom_type == "MultiPolygon":
                polygons.extend(g.geoms)
            else:
                polygons.append(g)
        return MultiPolygon(polygons)

    # Define aggregation dictionary
    # Use "first" for all, including aprland and aprbldg, except for geometry (collapsed)
    agg_dict = {col: "first" for col in gdf.columns if col not in (subset_cols + ["geometry"])}
    for val_col in ["aprland", "aprbldg"]:
        if val_col in gdf.columns:
            agg_dict[val_col] = "first"
    if "geometry" in gdf.columns:
        agg_dict["geometry"] = collapse_geometries

    gdf_collapsed = gdf.groupby(subset_cols, dropna=False).agg(agg_dict).reset_index()
    print(f"Collapsed dataframe now has {len(gdf_collapsed)} rows (from {len(gdf)} original rows).")

    # Make sure to preserve CRS and GeoDataFrame type if input was a GeoDataFrame
    from geopandas import GeoDataFrame

    # Note: Check if gdf is a GeoDataFrame (and keep old CRS)
    was_geodf = isinstance(gdf, GeoDataFrame)
    crs = gdf.crs if was_geodf and hasattr(gdf, "crs") else None
    # Only make a GeoDataFrame if geometry column is present
    if "geometry" in gdf_collapsed.columns:
        gdf_collapsed = GeoDataFrame(gdf_collapsed, geometry="geometry", crs=crs)

    # You could assign back if you want: gdf = gdf_collapsed
    gdf = gdf_collapsed.copy()

    # For completeness, print number of collapsed ( >1 handled) sets
    value_counts = gdf.groupby(subset_cols).size()
    n_multi = (value_counts > 1).sum()
    print(f"Collapsed {n_multi} sets of duplicate {subset_cols}.")

else:
    print(f"'dmp' column not found in gdf.")

# Find rows where 'dmp' == '07-06B-1' and display them
matching_rows = gdf[gdf['dmp'] == '07-06B-1']
display(matching_rows.head())

# Check the Coordinate Reference System (CRS) of the gdf
print("gdf CRS:", getattr(gdf, "crs", None))
import pandas as pd
pd.set_option('display.max_columns', None)
display(gdf.head())



PRC link: https://wvias.io/prc/Monongalia/2025/01%20%20%201000100000000
API url: https://api.wvias.io/v1/prc/Monongalia/2025/01%20%20%201000100000000?apiKey=REDACTED_WVIAS_API_KEY
  🏷️ Land Use: Residential 2 Family
PRC link: https://wvias.io/prc/Monongalia/2025/01%20%20%201000200000000
API url: https://api.wvias.io/v1/prc/Monongalia/2025/01%20%20%201000200000000?apiKey=REDACTED_WVIAS_API_KEY
  🏷️ Land Use: Residential 1 Family

Summary of API lookups for first two PRC links: ['Residential 2 Family', 'Residential 1 Family']
Processing batch 0 (0-1000), 4 to fetch.


Processing batch 1 (1000-2000), 3 to fetch.


Processing batch 2 (2000-3000), 8 to fetch.


Processing batch 3 (3000-4000), 7 to fetch.


Processing batch 4 (4000-5000), 5 to fetch.


...

,geometry,OBJECTID,dmp,parid,nbhd,own1,own2,careof,owneraddr,cityname,statecode,legal1,legal2,legal3,book,page,aprland,aprbldg,acres,dist,map,parcel,PRC,IAS_Details,DB_DP,Shape__Area,Shape__Length,LandUse
48209,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x0b\x00...,391770,07-06B-1,07 6B000100000000,470C,MONONGALIA COUNTY DEVELOPMENT,AUTHORITY,,1009 UNIVERSITY AVE,MORGANTOWN,WV,22.5912 AC SUR,HALL FARM,,1221,273,2078800.0,0.0,22.59,07,06B,1,https://wvias.io/prc/Monongalia/2025/07%20%206...,https://wvias.io/detail/Monongalia/2025/07%20%...,1221-273,3374.115377,271.246412,Vacant Exempt Land
48214,None,391775,07-06B-1,07 6B000100000000,470C,MONONGALIA COUNTY DEVELOPMENT,AUTHORITY,,1009 UNIVERSITY AVE,MORGANTOWN,WV,22.5912 AC SUR,HALL FARM,,1221,273,2078800.0,0.0,22.59,07,06B,1,https://wvias.io/prc/Monongalia/2025/07%20%206...,https://wvias.io/detail/Monongalia/2025/07%20%...,1221-273,NaN,NaN,Vacant Exempt Land
52369,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,395930,07-06B-1,07 6B000100000000,470C,MONONGALIA COUNTY DEVELOPMENT,AUTHORITY,,1009 UNIVERSITY AVE,MORGANTOWN,WV,22.5912 AC SUR,HALL FARM,,1221,273,2078800.0,0.0,22.59,07,06B,1,https://wvias.io/prc/Monongalia/2025/07%20%206...,https://wvias.io/detail/Monongalia/2025/07%20%...,1221-273,2.124066,16.289090,Vacant Exempt Land
52373,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x07\x00...,395934,07-06B-1,07 6B000100000000,470C,MONONGALIA COUNTY DEVELOPMENT,AUTHORITY,,1009 UNIVERSITY AVE,MORGANTOWN,WV,22.5912 AC SUR,HALL FARM,,1221,273,2078800.0,0.0,22.59,07,06B,1,https://wvias.io/prc/Monongalia/2025/07%20%206...,https://wvias.io/detail/Monongalia/2025/07%20%...,1221-273,1079.864109,270.409586,Vacant Exempt Land
52374,"b""\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...",395935,07-06B-1,07 6B000100000000,470C,MONONGALIA COUNTY DEVELOPMENT,AUTHORITY,,1009 UNIVERSITY AVE,MORGANTOWN,WV,22.5912 AC SUR,HALL FARM,,1221,273,2078800.0,0.0,22.59,07,06B,1,https://wvias.io/prc/Monongalia/2025/07%20%206...,https://wvias.io/detail/Monongalia/2025/07%20%...,1221-273,20.839539,417.817848,Vacant Exempt Land


Percent of records that intersect another geometry by more than 1.000% of their area: 1.98%

property_use_description counts for intersected parcels:
property_use_description
Condominium (Fee simple)          809
Residential 1 Family               57
Residential Vacant Land            54
Office Condominium                 37
None                               29
Vacant Exempt Land                 16
Active Farm                        14
Condominium (Common element)       13
General Commerical Vacant Land     13
Auxiliary Improvements              7
Office Bldg. - Low Rise (1-4 s      6
Apartment Vacant Land               5
Office/Warehouse                    4
Utility Vacant Land                 3
Mobile Home                         3
Other Miscellaneous Exempt          3
Vacant Land                         3
A                                   3
S                                   3
Religious                           3
Apartments Garden (1-3 stories      3
Parking Garage/Deck        

,dmp,OBJECTID,parid,nbhd,own1,own2,careof,owneraddr,cityname,statecode,legal1,legal2,legal3,book,page,aprland,aprbldg,acres,dist,map,parcel,PRC,IAS_Details,DB_DP,Shape__Area,Shape__Length,LandUse,geometry
15167,07-06B-1,391770,07 6B000100000000,470C,MONONGALIA COUNTY DEVELOPMENT,AUTHORITY,,1009 UNIVERSITY AVE,MORGANTOWN,WV,22.5912 AC SUR,HALL FARM,,1221,273,2078800.0,0.0,22.59,07,06B,1,https://wvias.io/prc/Monongalia/2025/07%20%206...,https://wvias.io/detail/Monongalia/2025/07%20%...,1221-273,3374.115377,271.246412,Vacant Exempt Land,"MULTIPOLYGON (((-80.02935 39.63367, -80.0293 3..."


gdf CRS: EPSG:4326


,dmp,OBJECTID,parid,nbhd,own1,own2,careof,owneraddr,cityname,statecode,legal1,legal2,legal3,book,page,aprland,aprbldg,acres,dist,map,parcel,PRC,IAS_Details,DB_DP,Shape__Area,Shape__Length,LandUse,geometry
0,--,391950,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,NaN,NaN,None,None,None,https://wvias.io/prc/Monongalia/2025/,https://wvias.io/detail/Monongalia/2025/,-,1.774707e+01,498.973138,None,"POLYGON ((-80.03249 39.71219, -80.03249 39.712..."
1,01-01-1,343561,01 1000100000000,1510,SHIPMAN R ALLAN,,,106 RENNER CREEK RD,NEW FREEPORT,PA,107.50 AC SUR,,,1506,314,105500.0,150200.0,107.5,01,01,1,https://wvias.io/prc/Monongalia/2025/01%20%20%...,https://wvias.io/detail/Monongalia/2025/01%20%...,1506-314,4.911763e+06,16840.497573,Residential 2 Family,"MULTIPOLYGON (((-80.4085 39.71459, -80.40953 3..."
2,01-01-2,343562,01 1000200000000,1510,"JOHNSTON LAWRENCE D,",MARY K JOHNSTON,,1017 DEREKTON DR,LATROBE,PA,95 AC SUR,DUNKARD CREEK,,1571,310,70000.0,20900.0,95.0,01,01,2,https://wvias.io/prc/Monongalia/2025/01%20%20%...,https://wvias.io/detail/Monongalia/2025/01%20%...,1571-310,4.378009e+06,11094.094029,Residential 1 Family,"MULTIPOLYGON (((-80.41494 39.71851, -80.4146 3..."
3,01-01-3,343563,01 1000300000000,1510,"JOHNSTON LAWRENCE D,",MARY K JOHNSTON,,1017 DEREKTON DR,LATROBE,PA,49.5 AC SUR,,,1571,310,31000.0,0.0,49.5,01,01,3,https://wvias.io/prc/Monongalia/2025/01%20%20%...,https://wvias.io/detail/Monongalia/2025/01%20%...,1571-310,1.885658e+06,7716.098597,Residential Vacant Land,"MULTIPOLYGON (((-80.41219 39.71654, -80.41208 ..."
4,01-01-4,343564,01 1000400000000,1510,PICHARDO PAUL G (II),,,1188 SAINT CLOUD RD,HUNDRED,WV,90 AC SUR & ALL COG MIRACLE,RUN (API #1572-CNX-CBM ONLY),,1356,490,30800.0,12400.0,90.0,01,01,4,https://wvias.io/prc/Monongalia/2025/01%20%20%...,https://wvias.io/detail/Monongalia/2025/01%20%...,1356-490,3.840050e+06,14200.046363,Active Farm,"MULTIPOLYGON (((-80.40601 39.71195, -80.40598 ..."


In [4]:
MORGANTOWN_LANDUSE_TO_CATEGORY = {
    # --- Residential ---
    "Residential 1 Family": "Single Family Residential",
    "Residential 2 Family": "Small Multi-Family (2–4 units)",
    "Residential 3 Family": "Small Multi-Family (2–4 units)",
    "Residential 4 Family": "Small Multi-Family (2–4 units)",
    "Apartments Garden (1-3 stories": "Large Multi-Family (5+ units)",
    "High Rise-Apartments": "Large Multi-Family (5+ units)",
    "Boarding/Rooming House": "Large Multi-Family (5+ units)",
    "Condominium (Fee simple)": "Other Residential",
    "Condominium (Common element)": "Other Residential",
    "Downtown Row Type": "Other Residential",
    "Residental Bldg. On Comm. Land": "Other Residential",
    "Res. Structure on Apt. Value L": "Other Residential",
    "Mixed Residental/Commercial": "Mixed Use",
    "Mixed Residential/Commercial": "Mixed Use",
    "Mobile Home": "Mobile Homes",
    "Mobile Home Park": "Mobile Homes",

    # --- Vacant land ---
    "Residential Vacant Land": "Vacant Land",
    "General Commerical Vacant Land": "Vacant Land",
    "Apartment Vacant Land": "Vacant Land",
    "Utility Vacant Land": "Vacant Land",
    "Lg. Vacant Tracts w/unknown": "Vacant Land",
    "Vacant Land": "Vacant Land",
    "Vacant Exempt Land": "Vacant Land",

    # --- Agriculture ---
    "Active Farm": "Agricultural",
    "Inactive Farm": "Agricultural",

    # --- Retail / service commercial ---
    "Retial - Single Occupancy": "Retail / Commercial",
    "Retail - Multiple Occupancy": "Retail / Commercial",
    "Restaurant": "Retail / Commercial",
    "Fast Food": "Retail / Commercial",
    "Convenience Food Market": "Retail / Commercial",
    "Supermarket": "Retail / Commercial",
    "Food Stand": "Retail / Commercial",
    "Strip Shopping Center": "Retail / Commercial",
    "Regional Shopping Mall": "Retail / Commercial",
    "Community Shopping Center": "Retail / Commercial",
    "Neighborhood Shopping Center": "Retail / Commercial",
    "Discount Department Store": "Retail / Commercial",
    "Department Store": "Retail / Commercial",
    "Hotel/Motel - Low Rise": "Retail / Commercial",
    "Hotel/Motel - High Rise": "Retail / Commercial",
    "Bar/Lounge": "Retail / Commercial",
    "Bank": "Retail / Commercial",
    "Savings Institution": "Retail / Commercial",
    "Auto Dealer - Full Service": "Retail / Commercial",
    "Auto Service Garage": "Retail / Commercial",
    "Service Station with Bays": "Retail / Commercial",
    "Service Station without Bays": "Retail / Commercial",
    "Car Wash - Automatic": "Retail / Commercial",
    "Car Wash - Manual": "Retail / Commercial",
    "Truck Stop": "Retail / Commercial",

    # --- Office ---
    "Office Bldg. - Low Rise (1-4 s": "Office",
    "Office Bldg. - High Rise (>4 s": "Office",
    "Office Condominium": "Office",
    "Medical Office": "Office",
    "Vetrinary Clinic": "Office",

    # --- Industrial / manufacturing ---
    "Warehouse": "Industrial / Manufacturing",
    "Mini Warehouse": "Industrial / Manufacturing",
    "Warehouse Prefabricated": "Industrial / Manufacturing",
    "Office/Warehouse": "Industrial / Manufacturing",
    "Manufacturing": "Industrial / Manufacturing",
    "Machinery & Equipment Mfg.": "Industrial / Manufacturing",
    "Newspaper Plant": "Industrial / Manufacturing",
    "Steam Generating Plant": "Industrial / Manufacturing",
    "Concrete Mfg.": "Industrial / Manufacturing",
    "Research & Development": "Industrial / Manufacturing",
    "Metal Working": "Industrial / Manufacturing",
    "Chemical Plant": "Industrial / Manufacturing",
    "Petroleum Refinery": "Industrial / Manufacturing",
    "Electronic Equipment Mfg.": "Industrial / Manufacturing",
    "Meat Packing & Slaughterhouse": "Industrial / Manufacturing",
    "Cold Storage Facility": "Industrial / Manufacturing",
    "Natural Gas Extracting Facilit": "Industrial / Manufacturing",
    "Glass Mfg.": "Industrial / Manufacturing",
    "Woodworking Shop": "Industrial / Manufacturing",
    "Saw Mills - Temporary": "Industrial / Manufacturing",
    "Mining, Deep": "Industrial / Manufacturing",
    "Mining, Strip": "Industrial / Manufacturing",
    "Compressor Station (not Public": "Industrial / Manufacturing",
    "Oil & Gas Pipeline (not Public": "Industrial / Manufacturing",

    # --- Parking (split lots vs structures) ---
    "Parking Miscellaneous": "Parking Lots",
    "Parking Garage/Deck": "Parking Structures",

    # --- Utilities / infrastructure ---
    "Telephone Equipment Bldg.": "Utilities / Infrastructure",
    "Radio/TV Transmitter Building": "Utilities / Infrastructure",
    "Rail/Bus/Air Terminal": "Utilities / Infrastructure",
    "Truck Terminal": "Utilities / Infrastructure",
    "Hangar": "Utilities / Infrastructure",
    "Utility Vacant Land": "Vacant Land",  # (already above)

    # --- Institutional / civic ---
    "Religious": "Institutional / Civic",
    "College & University": "Institutional / Civic",
    "School": "Institutional / Civic",
    "Library": "Institutional / Civic",
    "Cemetery": "Institutional / Civic",
    "Police or Fire Station": "Institutional / Civic",
    "Post Office": "Institutional / Civic",
    "Federal/State Building": "Institutional / Civic",
    "Hospital": "Institutional / Civic",
    "Nursing Home": "Institutional / Civic",
    "Day Care Center": "Institutional / Civic",
    "Funeral Home": "Institutional / Civic",

    # --- Recreation / cultural ---
    "Recreational/Health": "Recreation / Cultural",
    "Health Spa": "Recreation / Cultural",
    "Tennis Club - Indoor": "Recreation / Cultural",
    "Racquet Club - Indoor": "Recreation / Cultural",
    "Country Club (with Golf Course": "Recreation / Cultural",
    "Country Club (w/o Golf Course)": "Recreation / Cultural",
    "Club House": "Recreation / Cultural",
    "Amusement Park": "Recreation / Cultural",
    "Auditorium": "Recreation / Cultural",
    "Cultural": "Recreation / Cultural",
    "Bowling Alley": "Recreation / Cultural",
    "Motion Picture Theater": "Recreation / Cultural",
    "Legitimate Theater": "Recreation / Cultural",
    "Radio TV or Motion Picture Stu": "Recreation / Cultural",

    # --- Exempt / misc / odd codes ---
    "Auxiliary Improvements": "Exempt / Misc",
    "Auxiliary Improvement": "Exempt / Misc",
    "Other Miscellaneous Exempt": "Exempt / Misc",
    "Unsound Residential Structure": "Exempt / Misc",
    "Unsound Commercial Structure": "Exempt / Misc",
    "Ice House": "Exempt / Misc",
    "None": "Other",
    "A": "Other",
    "F": "Other",
    "S": "Other",
    "Socla/Fraternal Hall": "Institutional / Civic",  # keep here vs Recreation; easy to change
}
import html

def morgantown_category_from_landuse(x):
    # Ensure anything with 'exempt' (case-insensitive) in the original land use is classified as Exempt / Misc
    if x is None:
        return "Other"
    x_str = html.unescape(str(x)).strip()
    if "exempt" in x_str.lower():
        return "Exempt / Misc"
    return MORGANTOWN_LANDUSE_TO_CATEGORY.get(x_str, "Other")

gdf["PROPERTY_CATEGORY"] = gdf["LandUse"].apply(morgantown_category_from_landuse)
print(gdf["PROPERTY_CATEGORY"].value_counts(dropna=False))
vacant_land = gdf[gdf["PROPERTY_CATEGORY"].str.contains("Vacant", case=False, na=False)]
if "aprland" in vacant_land.columns:
    total = len(vacant_land)
    zero_aprland = (vacant_land["aprbldg"] <= 100).sum()
    percent_zero = zero_aprland / total * 100 if total > 0 else float('nan')
    print(f"Percentage of vacant land parcels with aprland=0: {percent_zero:.2f}%")
else:
    print("Column 'aprland' not found in the DataFrame.")

if "cityname" in gdf.columns:
    print(gdf["cityname"].value_counts(dropna=False))
else:
    print("Column 'cityname' not found in the DataFrame.")

# Pull the geography of Morgantown using boundary data from an open data portal or OSM.
# We'll use OSMnx to get the city limits for Morgantown, WV.

import osmnx as ox

# Get the polygon boundary for Morgantown, WV
morgantown_boundary = ox.geocode_to_gdf("Morgantown, West Virginia, USA")

print(morgantown_boundary)


from shapely.geometry import shape

# Ensure the gdf's CRS matches the boundary CRS
if gdf.crs is None:
    gdf = gdf.set_crs("EPSG:4326", allow_override=True)
if morgantown_boundary.crs != gdf.crs:
    morgantown_boundary = morgantown_boundary.to_crs(gdf.crs)

# Restrict gdf to those geometries inside morgantown_boundary (use first polygon)
boundary_geom = morgantown_boundary.geometry.iloc[0]

# Handle null geometries safely
gdf_inside = gdf[gdf["geometry"].apply(lambda x: x is not None and shape(x).is_valid) if hasattr(gdf.iloc[0]["geometry"], "__geo_interface__") else gdf["geometry"].notnull()]
gdf_inside = gdf_inside[gdf_inside["geometry"].apply(lambda geom: geom.within(boundary_geom) if geom is not None else False)]

n_total = len(gdf)
n_inside = len(gdf_inside)
percent_inside = 100 * n_inside / n_total if n_total > 0 else 0
print(f"{n_inside} rows ({percent_inside:.2f}% of total) are within the Morgantown boundary.")

gdf = gdf_inside.copy()

import numpy as np

# Create export dataframe
export_gdf = gdf.copy()

# Create exemption flag as binary (1/0) for fully exempt properties
# Filter out exempt properties (assume "Exempt" in PROPERTY_CATEGORY means exempt)
export_gdf = export_gdf[~export_gdf['PROPERTY_CATEGORY'].str.contains("Exempt", na=False)]

# Map property category (use existing PROPERTY_CATEGORY column from above)
export_gdf['property_land_use_category'] = export_gdf['PROPERTY_CATEGORY']

# Create refined property/land use category with three options: Vacant, Parking Lot, Underdeveloped
def categorize_property_refined(row):
    """Categorize properties into refined categories based on Morgantown land use/appraisal values."""
    category = row.get('PROPERTY_CATEGORY', None)
    if category is not None and 'Vacant' in str(category):
        return 'Vacant'
    elif category is not None and 'Parking' in str(category):
        return 'Parking Lot'
    elif ('aprbldg' in row) and ('aprland' in row):
        try:
            aprbldg = float(row.get('aprbldg', np.nan))
            aprland = float(row.get('aprland', np.nan))
            total = aprbldg + aprland
            if total > 0 and aprbldg < 0.5 * total:
                return 'Underdeveloped'
        except Exception:
            pass
    return None  # null for all other categories

export_gdf['property_land_use_refined'] = export_gdf.apply(categorize_property_refined, axis=1)

# Calculate area in square feet directly from projected geometry (do NOT use Shape__Area)
import geopandas as gpd

# Use WV North (US Survey feet) for accurate area (per @file_context_0)
feet_crs = "EPSG:6448"  # NAD83(WV North), US Survey Feet

if export_gdf.crs is None:
    export_gdf = export_gdf.set_crs("EPSG:4326", allow_override=True)

# Reproject to feet-based CRS for area calculation
export_gdf_ft = export_gdf.to_crs(feet_crs)

# Calculate area in sq ft (geometry may sometimes be None)
export_gdf['area_sqft'] = export_gdf_ft['geometry'].apply(
    lambda geom: geom.area if geom is not None else np.nan
)

# Calculate current tax per square foot
if 'current_tax' in export_gdf.columns:
    export_gdf['current_tax_per_sqft'] = np.where(
        export_gdf['area_sqft'] > 0,
        export_gdf['current_tax'] / export_gdf['area_sqft'],
        0
    )
else:
    export_gdf['current_tax'] = np.nan
    export_gdf['current_tax_per_sqft'] = np.nan

# Land value and improvement value: Morgantown uses aprland and aprbldg
export_gdf['land_value'] = export_gdf['aprland']
export_gdf['improvement_value'] = export_gdf['aprbldg']

# Calculate land value per square foot
export_gdf['land_value_per_sqft'] = np.where(
    export_gdf['area_sqft'] > 0,
    export_gdf['land_value'] / export_gdf['area_sqft'],
    0
)

# Calculate improvement value per square foot
export_gdf['improvement_value_per_sqft'] = np.where(
    export_gdf['area_sqft'] > 0,
    export_gdf['improvement_value'] / export_gdf['area_sqft'],
    0
)
export_gdf['link'] = export_gdf['PRC']
# Select columns for export, matching structure
columns_to_export = [
    'geometry',
    'property_land_use_category',
    'property_land_use_refined',
    'current_tax',
    'current_tax_per_sqft',
    'land_value',
    'land_value_per_sqft',
    'improvement_value',
    'improvement_value_per_sqft',
    'area_sqft',
    'link'
]

# Rename columns to match convention: land_value -> current_full_land_value etc
export_final = export_gdf[columns_to_export].rename(columns={
    'land_value': 'current_full_land_value'
})

# Ensure geometry is valid
export_final['geometry'] = export_final['geometry'].apply(
    lambda geom: geom if geom is None or geom.is_valid else geom.buffer(0)
)

# Ensure output is in WGS84 (EPSG:4326) before saving
if export_final.crs is None or export_final.crs.to_epsg() != 4326:
    export_final = export_final.set_crs(export_gdf.crs, allow_override=True)
    export_final = export_final.to_crs("EPSG:4326")
    print("Converted to EPSG:4326")

# Save as Parquet
output_filename = os.path.expanduser("~/Downloads/morgantown.parquet")
export_final.to_parquet(output_filename, index=False)

print(f"\n✅ Saved morgantown.parquet to Downloads")
print("Saved columns:", export_final.columns.tolist())
print("Property refined category counts:")
print(export_final['property_land_use_refined'].value_counts(dropna=False))
print("Property category counts:")
print(export_final['property_land_use_category'].value_counts().head(10))

# Display first few rows
print(f"\n👀 First 5 rows of exported data:")
display(export_final.head())

# View (display) export_gdf row(s) where dmp == '11-26-43'
rows = export_gdf[export_gdf['dmp'] == '11-26-43']
if rows.empty:
    print("No row found with dmp == '11-26-43'")
else:
    display(rows)

# Find the row where 'dmp' == '07-06B-1'
matching_row = export_gdf[export_gdf['dmp'] == '07-06B-1']
if not matching_row.empty:
    geom = matching_row.iloc[0]['geometry']
    # Ensure geometry is not None and is a (Multi)Polygon
    if geom is not None:
        gdf_tmp = gpd.GeoDataFrame(geometry=[geom], crs=export_gdf.crs)
        # Reproject to a feet-based CRS for accurate area measurement
        feet_crs = "EPSG:6448"  # NAD83(WV North), US Survey Feet
        gdf_tmp_ft = gdf_tmp.to_crs(feet_crs)
        area_sqft = gdf_tmp_ft.iloc[0].geometry.area
        print(f"Area of MULTIPOLYGON with dmp '07-06B-1': {area_sqft:,.2f} sq ft")
    else:
        print("No geometry found in the matching row.")
else:
    print("No row found with dmp == '07-06B-1'")



PROPERTY_CATEGORY
Single Family Residential         26120
Vacant Land                       15314
Exempt / Misc                      2716
Mobile Homes                       1914
Agricultural                       1885
Small Multi-Family (2–4 units)     1778
Other Residential                  1089
Other                              1008
Retail / Commercial                 662
Institutional / Civic               610
Industrial / Manufacturing          549
Large Multi-Family (5+ units)       514
Mixed Use                           297
Office                              282
Parking Lots                        184
Recreation / Cultural                92
Utilities / Infrastructure           23
Parking Structures                   12
Name: count, dtype: int64
Percentage of vacant land parcels with aprland=0: 98.92%
cityname
MORGANTOWN        38821
WESTOVER           1802
MAIDSVILLE         1317
FAIRVIEW           1257
FAIRMONT            828
                  ...  
NEW KENT              1
SI

,geometry,property_land_use_category,property_land_use_refined,current_tax,current_tax_per_sqft,current_full_land_value,land_value_per_sqft,improvement_value,improvement_value_per_sqft,area_sqft,link
19329,"POLYGON ((-79.95966 39.66374, -79.95947 39.663...",Vacant Land,Vacant,NaN,NaN,22400.0,2.356684,0.0,0.000000,9504.882553,https://wvias.io/prc/Monongalia/2025/08%20%20%...
22409,"POLYGON ((-79.90553 39.64481, -79.90559 39.644...",Vacant Land,Vacant,NaN,NaN,300.0,4.934361,0.0,0.000000,60.798148,https://wvias.io/prc/Monongalia/2025/08%20%20%...
22837,"POLYGON ((-79.93733 39.63863, -79.93728 39.638...",Vacant Land,Vacant,NaN,NaN,200.0,5.894234,0.0,0.000000,33.931467,https://wvias.io/prc/Monongalia/2025/08%20%20%...
23217,"POLYGON ((-79.9286 39.63756, -79.92875 39.6373...",Single Family Residential,Underdeveloped,NaN,NaN,62700.0,30.493505,48600.0,23.636114,2056.175592,https://wvias.io/prc/Monongalia/2025/08%20%208...
23218,"POLYGON ((-79.92841 39.63749, -79.92856 39.637...",Small Multi-Family (2–4 units),Underdeveloped,NaN,NaN,60900.0,88.854805,52200.0,76.161261,685.387808,https://wvias.io/prc/Monongalia/2025/08%20%208...


No row found with dmp == '11-26-43'
No row found with dmp == '07-06B-1'


In [9]:
from parcel_calculations import smooth_land_value_per_sqft
import os

# Ensure feet_crs is available for distance calculations
try:
    feet_crs
except NameError:
    feet_crs = "EPSG:6448"  # NAD83(WV North), US Survey Feet

# Smooth land value per sqft using distance-weighted 10 nearest neighbors
export_gdf["smooth_land_value_per_sqft"] = smooth_land_value_per_sqft(
    export_gdf,
    land_value_col="land_value",
    area_sqft_col="area_sqft",
    distance_crs=feet_crs,
    k=10,
)

export_gdf["smooth_full_land_value"] = (
    export_gdf["smooth_land_value_per_sqft"] * export_gdf["area_sqft"]
)

# Rebuild export with smooth columns and re-save
columns_to_export = [
    "geometry",
    "property_land_use_category",
    "property_land_use_refined",
    "current_tax",
    "current_tax_per_sqft",
    "land_value",
    "land_value_per_sqft",
    "smooth_full_land_value",
    "smooth_land_value_per_sqft",
    "improvement_value",
    "improvement_value_per_sqft",
    "area_sqft",
    "link",
]

export_final = export_gdf[columns_to_export].rename(
    columns={"land_value": "current_full_land_value"}
)

# Ensure output is in WGS84 (EPSG:4326) before saving
if export_final.crs is None or export_final.crs.to_epsg() != 4326:
    export_final = export_final.set_crs(export_gdf.crs, allow_override=True)
    export_final = export_final.to_crs("EPSG:4326")
canonical_path = os.path.join(data_dir, "morgantown-wv-parcels.parquet")
today_str = datetime.now().strftime("%Y_%m_%d")
dated_path = os.path.join(data_dir, f"morgantown-wv-parcels_{today_str}.parquet")

export_final.to_parquet(canonical_path, index=False)
export_final.to_parquet(dated_path, index=False)

print(f"✅ Saved export parquet: {canonical_path}")
print(f"✅ Also saved dated version: {dated_path}")
print("Export columns:", export_final.columns.tolist())
print("\nAdded smooth land value columns:")
print([c for c in export_final.columns if "smooth_" in c])




✅ Saved export parquet: data/morgantown/morgantown-wv-parcels.parquet
✅ Also saved dated version: data/morgantown/morgantown-wv-parcels_2026_01_23.parquet
Export columns: ['geometry', 'property_land_use_category', 'property_land_use_refined', 'current_tax', 'current_tax_per_sqft', 'current_full_land_value', 'land_value_per_sqft', 'smooth_full_land_value', 'smooth_land_value_per_sqft', 'improvement_value', 'improvement_value_per_sqft', 'area_sqft', 'link']

Added smooth land value columns:
['smooth_full_land_value', 'smooth_land_value_per_sqft']


In [10]:
# Optional: upload export_final to dev Azure blob
upload_dev = True

if upload_dev:
    from azure.storage.blob import BlobServiceClient

    connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
    if not connection_string:
        raise ValueError(
            "Set AZURE_STORAGE_CONNECTION_STRING or update connection_string before upload."
        )

    container = os.getenv("AZURE_DEV_CONTAINER", "parquets-dev")
    blob_name = "morgantown-wv-parcels.parquet"
    local_path = os.path.join(data_dir, blob_name)

    if not os.path.exists(local_path):
        raise FileNotFoundError(f"Local parquet not found: {local_path}")

    blob_service = BlobServiceClient.from_connection_string(connection_string)
    container_client = blob_service.get_container_client(container)

    with open(local_path, "rb") as handle:
        container_client.upload_blob(name=blob_name, data=handle, overwrite=True)

    print(f"✅ Uploaded {local_path} -> {container}/{blob_name}")
else:
    print("upload_dev is False; skipping upload.")


✅ Uploaded data/morgantown/morgantown-wv-parcels.parquet -> parquets-dev/morgantown-wv-parcels.parquet
